# Topology Optimisation with EllPHi Gradients

This notebook demonstrates how to use `ellphi.grad` for gradient-based
optimisation of ellipsoid configurations. No dependencies beyond **ellphi**
and **scipy** are required.

**Sections**
1. Gradient-aware tangency (single pair)
2. Batch gradients and the VJP pattern
3. [Placeholder] Persistence diagram optimisation

In [ ]:
import numpy as np
import ellphi
from ellphi import (
    tangency_grad,
    pdist_tangency_grad,
    pdist_tangency,
    coef_from_axes,
    coef_from_cov,
    tangency,
)

rng = np.random.default_rng(42)
print('ellphi version:', ellphi.__version__)

---
## 1. Gradient-aware tangency (single pair)

`tangency_grad(p, q)` returns a `TangencyGrad` dataclass with three fields:

| Field | Shape | Meaning |
|-------|-------|---------|
| `t` | scalar | Tangency distance (same as `tangency(p, q).t`) |
| `dt_dp` | `(m,)` | Gradient of `t` w.r.t. first-ellipsoid coefficients |
| `dt_dq` | `(m,)` | Gradient of `t` w.r.t. second-ellipsoid coefficients |

`coef_from_axes(X, r0, r1, theta)` builds a single ellipse coefficient vector
from a center `X`, semi-axes `r0`/`r1`, and orientation angle `theta`.

In [ ]:
# Build two 2-D ellipses from axes/angle representation
p = coef_from_axes(np.array([0.0, 0.0]), 3.0, 1.0, 0.3)
q = coef_from_axes(np.array([4.0, 1.0]), 2.0, 0.5, -0.5)

g = tangency_grad(p, q)
print(f't          = {g.t:.6f}')
print(f'dt_dp      = {g.dt_dp}')
print(f'dt_dq      = {g.dt_dq}')

In [ ]:
# Quick finite-difference check for dt_dp[0]
h = 1e-6
p_plus = p.copy()
p_plus[0] += h
p_minus = p.copy()
p_minus[0] -= h
fd = (tangency(p_plus, q).t - tangency(p_minus, q).t) / (2 * h)
print(f'analytic dt_dp[0] = {g.dt_dp[0]:.8f}')
print(f'finite-diff       = {fd:.8f}')
print(f'relative error    = {abs(g.dt_dp[0] - fd) / abs(fd):.2e}')

---
## 2. Batch gradients and the VJP pattern

`pdist_tangency_grad(coefs)` returns:
- `dists`: condensed pairwise distance array (same as `pdist_tangency`)
- `vjp`: a pullback `grad_dists → grad_coefs` that accumulates upstream
  gradients into per-ellipsoid coefficient gradients

The VJP can be chained with any scalar loss `L(dists)` via the chain rule:
`∂L/∂coefs = vjp(∂L/∂dists)`

In [ ]:
# Build a small cloud of N=5 random 2-D ellipses
N = 5
means = rng.uniform(-20, 20, (N, 2))
covs = np.stack([
    (lambda F: F @ F.T + 4 * np.eye(2))(rng.standard_normal((2, 2)))
    for _ in range(N)
])
coefs = coef_from_cov(means, covs)

dists, vjp = pdist_tangency_grad(coefs)

# Verify distances match pdist_tangency
ref = pdist_tangency(coefs)
print('max |dists - ref|:', np.max(np.abs(dists - ref)))
print('distances:', dists)

In [ ]:
# Example: gradient of sum(dists) w.r.t. all coefficients
grad_coefs = vjp(np.ones_like(dists))
print('grad_coefs shape:', grad_coefs.shape)  # (N, m)
print('grad_coefs[0]:', grad_coefs[0])

In [ ]:
# Wiring into scipy.optimize.minimize(jac=True, method='L-BFGS-B')
# would look like the following.  The choice of loss L and the
# parameterisation of coefs (e.g. centre-only vs full) depends on the
# specific optimisation problem.

# def objective_and_grad(x):
#     coefs = x.reshape(N, m)
#     dists, vjp = pdist_tangency_grad(coefs)
#     loss = L(dists)                        # user-supplied scalar loss
#     grad_coefs = vjp(dL_ddists(dists))     # upstream gradient
#     return float(loss), grad_coefs.ravel()
#
# from scipy.optimize import minimize
# result = minimize(objective_and_grad, coefs.ravel(),
#                   method='L-BFGS-B', jac=True)

---
## 3. [Placeholder] Persistence diagram optimisation

This section shows *where* to insert TDA code to drive optimisation via
persistence diagrams. The gradient hookup via `vjp` is already shown in
Section 2 — only the TDA loss and its gradient need to be supplied.

```python
# ── USER-SUPPLIED TDA CODE ──────────────────────────────────────────────────
# import your_tda_library as tda
#
# def persistence_loss_and_grad(dists, n_ellipses):
#     """Compute TDA loss and its gradient w.r.t. pairwise distances.
#
#     Args:
#         dists:      condensed pairwise tangency-distance array (n_pairs,)
#         n_ellipses: number of ellipses N
#
#     Returns:
#         loss:       scalar
#         grad_dists: (n_pairs,) gradient of loss w.r.t. dists
#     """
#     dgm = tda.rips_persistence(dists, n_ellipses)
#     loss = tda.bottleneck_distance(dgm, target_dgm)
#     grad_dists = tda.bottleneck_gradient(dgm, target_dgm)
#     return loss, grad_dists
# ────────────────────────────────────────────────────────────────────────────
#
# def tda_objective_and_grad(x):
#     coefs = x.reshape(N, m)
#     dists, vjp = pdist_tangency_grad(coefs)
#     loss, grad_dists = persistence_loss_and_grad(dists, N)
#     return float(loss), vjp(grad_dists).ravel()
#
# result = minimize(tda_objective_and_grad, coefs.ravel(),
#                   method='L-BFGS-B', jac=True)
```